> **公开学习副本**：保留教学代码与解释，已清除运行输出、执行计数、嵌入附件及个人元数据。完整执行原件和H02交接包保留在本地/组内私有空间。不要把本副本误认为团队已验收的新数据版本。
>
> 数据来自Olist；基础教学材料参考[CercaTrovato/olist-repurchase-analysis](https://github.com/CercaTrovato/olist-repurchase-analysis)，保留上游代码与数据许可。仅在本地有原始数据时运行；输出可能包含逐行客户数据，保持在被Git忽略的outputs目录。源码中的定义及限制仍应阅读。
> 历史学习课：用于理解问题，不取代当前H01/H02。窗口对比只针对已开放的三期，不要用它探索待封存的最终标签。

# Olist 从头做项目 · 第1课：拿到数据，先判断它是什么

今天的目标：能解释每张表的一行代表什么、表怎样连接、数据能回答什么问题。

这是独立学习副本，不会修改小组 GitHub。数据来自你们仓库原始包：
https://github.com/CercaTrovato/olist-repurchase-analysis
数据发布方 Olist；许可 CC BY-NC-SA 4.0。金额保留原始口径，当前不把商品金额叫利润。

## 全项目路线
1. 数据理解：来源、表结构、粒度、键、时间范围、质量。
2. 业务定义：服务谁、预测什么、预测时点、什么叫复购。
3. 清理与关联：先统一订单粒度，再形成客户历史记录。
4. EDA：看订单和客户分布，找问题与候选特征。
5. 客户快照：过去六个月的 X、未来90天的 y。
6. 时间划分与基准：训练/验证/测试，先比较简单规则。
7. 分类建模与评价：比较模型，解释不平衡数据下的指标。
8. 聚类与逐群 EDA：找群体差异，提出小规模特征实验。
9. 验证改动与业务解释：冻结方法、评价局限、形成建议。
10. PPT、报告、问答：用证据回答业务问题。

这不是要求你今天跑完十步。今天只完成第1步。每格先读问题、猜结果，再运行并解释。

## 0. 找到原始数据
本 notebook 与 data 文件夹放在同一目录。代码直接读取 ZIP，不需要解压，也不需要训练库。
如缺 pandas，请在 notebook 新格运行 `%pip install pandas`。

现在开始第一阶段。

拿到数据，第一件事是弄清“一行代表什么”

假设有人给你一个压缩包，你还不知道研究题目。先不要想“用哪个模型”，先问：
1. 数据来自哪里，记录了什么业务？
2. 有几张表，各有哪些字段？
3. 每张表的一行代表什么？
4. 哪些字段能关联这些表？
5. 时间范围、缺失、重复和异常是什么情况？
6. 根据这些信息，哪些问题值得继续研究？
我实际读取 Olist 原始包，得到：
| 表 | 行数 | 初步理解 |
|---|---:|---|
| orders | 99,441 | 订单 |
| customers | 99,441 | 客户资料，通过订单侧客户编号关联 |
| order_items | 112,650 | 订单内的商品明细 |
| payments | 103,886 | 支付记录 |
| reviews | 99,224 | 评价记录 |
| products | 32,951 | 商品属性 |
| sellers | 3,095 | 卖家属性 |
| geolocation | 1,000,163 | 地理记录 |
| category translation | 71 | 品类翻译 |


这些数字不能直接相加，也不能把每张表的行数都叫“样本数量”。 它们记录的是不同业务对象。

例如，商品明细有112,650行，订单有99,441行，可能是因为一笔订单包含多条商品明细。这需要检查关联键来确认。

我们已经发现三个会影响后续分析的问题

第一，两个客户编号有不同用途。

真实检查结果：
customer_id 的不同值：       99,441
customer_unique_id 的不同值：96,096
你们要通过 customer_id 连接订单与客户资料，再用 customer_unique_id 追踪同一个买家的购买记录。

如果一开始就把 customer_id 当成跨订单的买家身份，复购统计就可能做错。

第二，商品明细行数不等于订单数。

我选出一笔真实订单：
明细1：商品金额 49.90
明细2：商品金额 49.90
它表示：
订单数：1
明细行数：2
商品金额合计：99.80
这会直接影响后面的特征：
- 计算购买频次，要数不同订单；
- 计算商品消费金额，要按正确粒度求和；
- 连接表后，不能直接把行数当作订单数。
第三，日期存在不代表观察期完整。

原始订单购买时间覆盖2016年9月至2018年10月，但：
2018年8月：6,512单
2018年9月：   16单
2018年10月：   4单
我们目前只能确认文件中这些月份的记录数量，不能直接断言订单量暴跌，也不能认定末尾月份覆盖完整。

这对复购预测很关键：如果客户后续90天没有被完整观察，不能轻率地把“没有记录”当成“不复购”。

另外，批准时间有160条缺失。现在先记录这个发现，之后根据“合格订单”的定义决定怎样处理，而不是一看到缺失就删掉整行。

为什么看完这些，才开始确定研究问题？

因为它们告诉我们：
数据有买家标识、订单时间和金额，因此具备研究购买行为及复购的基础。

但还需要继续验证后续观察期、复购人数和订单口径。我们因此把小组题目作为一个需要检验的方案：
利用已有客户过去六个月的购买行为，预测未来90天是否出现新的合格订单。

我们暂时还不能用这份数据证明“某种营销措施能增加复购”，因为没有相应的营销实验记录。

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/olist-v2.zip').is_file()), Path.cwd())
DATA_ZIP = ROOT / 'data' / 'olist-v2.zip'
assert DATA_ZIP.exists(), '请把当前工作目录切换到本 notebook 所在的 olist_learning 文件夹'
expected = '967e41e04fc306fe604e2a693f488995a8b41e5047418f8a5c8e4abd6deca784'
assert hashlib.sha256(DATA_ZIP.read_bytes()).hexdigest() == expected
print('原始包校验通过。pandas 版本：', pd.__version__)

def read_table(name):
    with ZipFile(DATA_ZIP) as z:
        with z.open(name) as f:
            return pd.read_csv(f)


# 显式加载，避免依赖上一次内核留下的变量。
customers = read_table("olist_customers_dataset.csv")
orders = read_table("olist_orders_dataset.csv")
items = read_table("olist_order_items_dataset.csv")


## 1. 不急着选算法，先盘点文件
问题：这是“一张整理好的机器学习表”，还是需要连接的业务表？
逐张读取并记录行列数。循环每次只保留一张临时表。
`shape` 给出（行数，列数）；列名帮助我们初步认识业务含义。

In [ ]:
with ZipFile(DATA_ZIP) as z:
    names = sorted(n for n in z.namelist() if n.endswith('.csv'))
inventory = []
for name in names:
    table = read_table(name)
    inventory.append({'table': name, 'rows': len(table), 'columns': table.shape[1]})
    print(name)
    print('字段：', ', '.join(table.columns))
    print()
print(pd.DataFrame(inventory).to_string(index=False))
del table


In [ ]:
print("客户表行数：", len(customers))
print("customer_id 不同值：",
      customers["customer_id"].nunique())
print("customer_unique_id 不同值：",
      customers["customer_unique_id"].nunique())

In [ ]:
# 找到一个在客户表中出现多次的买家编号
counts = customers["customer_unique_id"].value_counts()
buyer = counts[counts > 1].index[0]

# 查看这个买家对应哪些 customer_id
print(
    customers.loc[
        customers["customer_unique_id"] == buyer,
        ["customer_id", "customer_unique_id"]
    ]
)

In [ ]:
linked = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    validate="one_to_one"
)

print(
    linked.loc[
        linked["customer_unique_id"] == buyer,
        ["order_id", "customer_id",
         "customer_unique_id", "order_purchase_timestamp"]
    ]
)

怎样看出“明细行数不等于订单数”？

In [ ]:
print("商品明细行数：", len(items))
print("不同订单数：", items["order_id"].nunique())

In [ ]:
# 每个订单号对应多少行明细
counts = items.groupby("order_id").size()

# 选择一笔有多行明细的订单
example_id = counts[counts > 1].index[0]

detail = items.loc[
    items["order_id"] == example_id,
    ["order_id", "order_item_id", "price"]
]

print(detail)

怎样看出“日期很晚，不代表最后90天完整”？

这点需要修正一下表述：我们发现的是覆盖完整性存在疑问，而不是已经证明数据缺失。

In [ ]:
purchase_time = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

print(purchase_time.min())
print(purchase_time.max())

最大值只能说明：
文件里至少存在一笔这么晚的订单。

它不能说明这一天以前的所有订单都被收录。

In [ ]:
monthly_orders = (
    orders.assign(month=purchase_time.dt.to_period("M"))
    .groupby("month")["order_id"]
    .nunique()
)

print(monthly_orders)

如果9—10月的订单没有完整收录，一个客户即使真实复购，也可能没有出现在文件里。把他标成 y=0，就可能标错。

所以我们的判断是：
在确认覆盖情况之前，不能仅凭最大订单日期，认定某个90天结果窗口可用。

另外，批准时间缺失160条，是另一个独立检查：

## 2. 先确定粒度：一行代表什么？

| 表 | 一行的业务含义 | 初步用途 |
|---|---|---|
| orders | 一笔订单 | 日期、状态、关联客户 |
| customers | 一个订单侧 customer_id 对应的客户资料 | 用 customer_unique_id 追踪同一买家 |
| order_items | 订单内一条商品明细 | 商品金额、运费、商品与卖家关联 |
| payments | 一笔订单的一条支付记录 | 支付方式与金额 |
| reviews | 一条评价记录，不能先假设每单唯一 | 评分与评论 |
| products | 一个商品的属性 | 品类、重量等 |
| sellers | 一个卖家的属性 | 卖家地域 |
| geolocation | 一条邮编前缀相关坐标记录，前缀可能重复 | 地理信息，需先处理粒度 |
| translation | 一项品类名称翻译 | 品类解释 |

先集中看前三张。`head()` 看几行不是完整分析，但能帮助识别记录结构。

In [ ]:
orders = read_table('olist_orders_dataset.csv')
customers = read_table('olist_customers_dataset.csv')
items = read_table('olist_order_items_dataset.csv')
print('订单样例：')
print(orders.head(3).to_string(index=False))
print('\n客户资料样例：')
print(customers.head(3).to_string(index=False))
print('\n商品明细样例：')
print(items.head(3).to_string(index=False))


## 3. 检查键：能否凭某一列识别一行？
`nunique()` 数不同值；`duplicated().sum()` 数重复行/键。
重点：customer_id 与 customer_unique_id 不能混用。我们通过前者连接订单，通过后者追踪买家复购。
订单与商品明细是一对多；直接连接后，一笔订单可能占多行，不能再用总行数当订单数。

In [ ]:
for name, table, key in [
    ('orders', orders, 'order_id'),
    ('customers', customers, 'customer_id'),
    ('items', items, 'order_id')
]:
    print(name, '总行数=', len(table), '键唯一值=', table[key].nunique(),
          '键缺失=', table[key].isna().sum(), '键重复=', table[key].duplicated().sum())
print('客户资料中的 customer_unique_id 数：', customers['customer_unique_id'].nunique())
print('商品明细复合键重复：', items.duplicated(['order_id', 'order_item_id']).sum())
print('整行重复：', {name: int(t.duplicated().sum()) for name, t in
                    [('orders', orders), ('customers', customers), ('items', items)]})
print('订单缺少客户资料的数量：', (~orders['customer_id'].isin(customers['customer_id'])).sum())
print('明细找不到订单的行数：', (~items['order_id'].isin(orders['order_id'])).sum())


## 4. 先看时间、状态、缺失和金额，再决定如何清理
不要见到缺失值就删除。尚未批准、尚未送达与记录错误是不同情况。
下面先诊断，不做清理。先保留原始日期字符串，再用 `errors='coerce'` 检查解析失败。
月度数量只说明文件中观察到的订单量，不能仅凭月份少就判断业务衰退或数据完整。


In [ ]:
date_cols = [c for c in orders.columns if 'timestamp' in c or c.startswith('order_approved') or c.startswith('order_delivered') or c.startswith('order_estimated')]
for col in date_cols:
    original = orders[col]
    parsed = pd.to_datetime(original, errors='coerce')
    bad = (original.notna() & parsed.isna()).sum()
    orders[col] = parsed
    print(col, '缺失=', parsed.isna().sum(), '解析失败=', bad,
          '范围=', parsed.min(), '至', parsed.max())
print('\n订单状态：')
print(orders['order_status'].value_counts(dropna=False).to_string())
monthly = orders.groupby(orders['order_purchase_timestamp'].dt.to_period('M')).size()
print('\n各月订单数：')
print(monthly.to_string())
print('\n商品价格与运费摘要：')
print(items[['price', 'freight_value']].describe().to_string())
print('\n三张表的缺失数：')
for name, t in [('orders', orders), ('customers', customers), ('items', items)]:
    print(name, t.isna().sum().to_dict())


## 5. 用一笔真实订单，理解“一单多行”
下面选第一笔有多条商品明细的订单。先预测：它有几单？商品金额应取某一行还是求和？
这是观察结构，不是在对全数据清理。

In [ ]:
counts = items.groupby('order_id').size()
example_id = counts[counts > 1].index[0]
example_items = items.loc[items['order_id'] == example_id]
print('订单记录：')
print(orders.loc[orders['order_id'] == example_id, ['order_id', 'customer_id', 'order_status']].to_string(index=False))
print('\n订单明细：')
print(example_items[['order_id', 'order_item_id', 'price', 'freight_value']].to_string(index=False))
print('不同订单数：', example_items['order_id'].nunique())
print('明细行数：', len(example_items))
print('商品金额合计：', round(example_items['price'].sum(), 2))
print('运费合计：', round(example_items['freight_value'].sum(), 2))


## 6. 根据证据判断：能回答什么？

已有客户标识、订单时间和金额，因此有理由继续检查复购预测的可行性。但还需检查：每个快照有没有足够的历史与后续90天记录、复购者数量够不够、标签口径是什么。

其他可能问题包括配送时长、评论评分、品类差异。没有营销干预分组，不能直接证明某营销方案提高了复购；没有完整成本，不能把商品金额当利润。

现在才将小组题目写成候选定义：
“对过去六个月有合格订单的客户，利用截至预测时点已知的行为，预测未来90天是否出现新合格订单。”
下一课将检查合格订单定义，并把商品明细先汇总到订单粒度再连接客户。

## 你来解释（回答比运行成功更重要）
1. 哪张表一行是一笔订单？哪张表一行是商品明细？
2. 为什么不能用 customer_id 直接统计同一个买家的复购？
3. 第5节里那笔订单有几条明细、几笔订单、商品金额多少？
4. 为什么订单日期最大值足够晚，仍不保证最后90天覆盖完整？
5. 我们能预测复购，是否就证明联系这些人能增加复购？

先完成前三题，再继续清理与连接。不要直接把本课所有表一次性 merge。


哪张表一行是一笔订单？哪张表一行是商品明细？表	一行的业务含义	初步用途		
orders表的一行是一笔订单，order_items	是商品明细；		reviews	一条评价记录；

为什么不能用 customer_id 直接统计同一个买家的复购？因为customer_id不是customer_unique_id，一个用户可能有多个customer_id但只有一个customer_unique_id，所以要用customer_unique_id统计同一个买家的复购

第5节里那笔订单有几条明细、几笔订单、商品金额多少？1笔订单有2条订单明细，商品金额都是49.9总额99.8

为什么订单日期最大值足够晚，仍不保证最后90天覆盖完整？因为存在订单日期最大值足够晚但是6个月内订单不合格的客户这样吗？什么叫对过去六个月有合格订单的客户，是要求这个客户订单数量达到一定多吗？还是具体怎么判断呢？已有客户标识、订单时间和金额，因此有理由继续检查复购预测的可行性。但还需检查：每个快照有没有足够的历史与后续90天记录、复购者数量够不够、标签口径是什么。因为我们是监督预测不是无监督预测，

我们能预测复购，是否就证明联系这些人能增加复购？不能我们预测复购是基于已有数据集做出的合理假设，真实情况会受到更多条件与突发事情的影响，是不可能与预测结果完全符合的

第一个是：哪些客户有资格进入预测名单？

仓库当前的练习口径是：
在预测时点之前六个月内，至少有一笔“已批准、商品金额合计为正”的订单。

所以，不要求买很多次，有一笔符合条件的订单就可以。 “合格”指订单满足规则，不是客户消费达到某个等级。
例如，假设预测时点是2018年4月1日：
| 客户情况 | 是否进入这次预测名单？ |
|---|---|
| 2018年3月有一笔已批准、商品金额100的订单 | 是 |
| 过去六个月有三笔这样的订单 | 是 |
| 只在2017年5月买过，之后没有订单 | 否，不在这次历史窗口内 |
| 过去六个月只有未批准的订单 | 按当前口径不进入 |
“文件中至少有一条很晚的记录”，不等于“截至那一天，所有应有记录都被完整收集”。

如果未来记录不完整，把没有观察到订单的客户全部标成 y=0，就可能把“没有看到”误当成“没有复购”。

这与监督学习、无监督学习的区别无关，主要是数据覆盖与标签可靠性问题。 监督学习需要真实标签，但并不保证我们构造出来的标签一定可靠。

意思是前6个月有足够详细，但后面90天没有收录信息也不可以去完成这次的预测任务吗？

对，但要区分“做预测”和“训练、检验预测”。

过去六个月的信息足够，我们就可能构造输入 X；后续90天的记录决定我们能否知道真实答案 y。
| 场景 | 没有完整的后续90天记录，能不能做？ |
|---|---|
| 用已经训练好的模型，预测客户未来会不会复购 | **可以**。实际使用时，本来就不知道未来 |
| 把这些客户作为有标签的训练样本 | **不能直接使用**，因为答案可能不确定 |
| 用这些客户评价模型预测得准不准 | **不能可靠评价**，因为缺少完整真实结果 |
例如，一个客户过去六个月买了3单，但之后只观察了20天，没有看到新订单：
我们只知道“前20天没有观察到复购”，不知道“90天内不复购”。他可能第40天才购买。

所以，不能直接把他标成 y=0。

这也不意味着整个项目做不了。我们可以选择更早的预测时点，让历史六个月和未来90天都落在覆盖可靠的时期内：

90天不是一个日期，而是预测的时间跨度。

我们是怎样选出来的？需要同时考虑业务需求和数据条件。

第一，先根据业务确定“想预测多久”。

例如，运营团队如果每季度规划一次客户维护，那么“未来约一个季度是否复购”就是一个合理的候选问题。

但这只是设计理由的例子。你们目前采用90天，是仓库现有的研究设置；并不是数据自动算出90天最好，也不是必须选90天。最终需要小组确认。

选择30天、90天、180天，会得到不同的任务：
- 太短，可能观察到的复购者很少；
- 太长，需要更长的数据覆盖，而且预测离当前更远。
第二，再检查哪些日期适合当预测时点。

我们要保证这个日期的两边都有足够记录：

       过去6个月                 未来90天
├────────────────────┤────────────────────┤
历史开始                 预测时点                 结果窗口结束

2017年10月1日          2018年4月1日          2018年6月30日
├────计算历史特征 X─────┤────确定复购标签 y─────┤
窗口按“包含开始、不包含结束”处理。因此：
- 历史订单：`2017-10-01 ≤ 时间 < 2018-04-01`
- 未来订单：`2018-04-01 ≤ 时间 < 2018-06-30`

### 现在正式进入第2阶段。
定义问题不是先选算法，而是先把“我们到底要预测什么”写得足够具体。

我们先沿用仓库方案，形成一张练习版问题定义表：
| 必须明确的事项 | 本次练习的定义 |
|---|---|
| 谁会使用结果？ | 电商客户运营人员 |
| 要支持什么决策？ | 在联系资源有限时，辅助排列客户联系优先级 |
| 研究对象是谁？ | 过去六个月至少有一笔合格订单的已有客户 |
| 什么时候预测？ | 在一个明确的快照日，例如2018年4月1日 |
| 当时能用什么信息？ | 快照日前已经知道的购买行为 |
| 预测什么？ | 未来90天是否出现至少一笔新的合格订单 |
| 模型输出什么？ | 用于排序的复购分数；能否解释为真实概率需另行检查 |
| 怎样判断有用？ | 是否比简单规则更好地把后来复购的人排在前面 |
| 暂时不能证明什么？ | 联系客户能增加多少复购、收入或利润 |

在某个预测时点，利用过去六个月有合格订单的客户的历史购买行为，预测其未来90天是否再次下单，并检验模型的客户排序能力是否优于简单规则。

下来要做的是：将这段文字变成可执行的数据规则。 我们会先建立订单表，再依据规则筛选历史客户、计算特征和生成未来标签。

进入下一步前，用这个小例子检查两种窗口是否分清：

假设预测日是2018年4月1日，以下订单均符合资格：
| 客户 | 订单日期 |
|---|---|
| A | 2018年3月10日、2018年5月10日 |
| B | 2018年2月10日 |
| C | 2018年5月10日 |
    谁应该进入4月1日的预测名单？进入名单的客户，`y` 分别是几？ 这里最值得想清楚的是C：虽然他在未来买了，但预测当时我们是否已经有资格把他放进“历史客户”的名单。